# 02 · 무손실 압축 — 의미를 하나도 버리지 않고 어디까지 줄일 수 있을까요

01번에서 **무엇이 과금되는지** 확인했습니다. 이번에는 **실제로 줄여봅니다.**

### 왜 무손실부터 시작할까요

| | |
|---|---|
| **판단 기준이 됩니다** | 무손실로 40% 줄었다면, 손실 압축을 도입할 이유가 그만큼 줄어듭니다 |
| **위험이 없습니다** | 정보를 버리지 않으므로 답이 틀려질 수 없습니다 |
| **증명할 수 있습니다** | 압축 결과를 되돌려 원본과 대조하면 무손실임이 확인됩니다 |

> 이 단계를 건너뛰고 LLMLingua 부터 붙이면, 나중에 **"그거 안 써도 됐는데"** 가 됩니다.

### 다룰 변환 4가지

| # | 변환 | 무손실 여부 | 원리 |
|---|---|---|---|
| 1 | **정규화** | 무손실 | 주석·구분선·중복 공백을 제거합니다 |
| 2 | **구조 변환** | 무손실 | JSON 배열을 표로 바꿔 **키 반복을 없앱니다** |
| 3 | **공통값 추출** | 무손실 | 모든 행이 같은 필드를 헤더로 뺍니다 |
| 4 | **화이트리스트 컷** | **손실** | 안 쓰는 필드를 버립니다 (경계선) |

1~3은 되돌릴 수 있고, 4는 되돌릴 수 없습니다. **그 경계를 분명히 보는 것**이 이 노트북의 목적입니다.

### 준비물
- `az login` · 커널 `Python 3.12 (.venv · token-compression)` · 이 폴더의 `.env`

## 1. 설정

01번과 동일합니다. 엔드포인트와 배포명은 `.env` 에서 읽습니다.

In [ ]:
import json, os, re, subprocess, sys, shutil, time
import urllib.request, urllib.error
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

# 노트북 공용 유틸 (같은 폴더의 nbtools.py) — 01번과 동일
from nbtools import Usage, Price, show_table

ENV_PATH = find_dotenv(usecwd=True) or str(Path.cwd() / ".env")
load_dotenv(ENV_PATH, override=False)


def require(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(f"{name} 가 없습니다. `cp .env.example .env` 후 값을 채우세요.")
    return v


ENDPOINT   = require("AZURE_OPENAI_ENDPOINT").rstrip("/")
DEPLOYMENT = require("AZURE_OPENAI_DEPLOYMENT")

AZ_CANDIDATES = [os.environ.get("AZ_CLI"), shutil.which("az"),
                 "/opt/homebrew/bin/az", "/usr/local/bin/az",
                 str(Path.home() / ".local/bin/az")]


def find_az():
    for c in AZ_CANDIDATES:
        if c and Path(c).exists():
            return c
    raise RuntimeError("az CLI 를 찾지 못했습니다. .env 에 AZ_CLI=/전체/경로/az 를 넣으세요.")


def auth_headers():
    key = os.environ.get("AZURE_OPENAI_API_KEY")
    if key:
        return {"api-key": key}
    r = subprocess.run([find_az(), "account", "get-access-token",
                        "--scope", "https://cognitiveservices.azure.com/.default", "-o", "json"],
                       capture_output=True, text=True, timeout=90)
    if r.returncode != 0:
        raise RuntimeError(f"az 토큰 발급 실패 (`az login` 필요?)\n{r.stderr.strip()[:300]}")
    return {"Authorization": "Bearer " + json.loads(r.stdout)["accessToken"]}


HEADERS = auth_headers()


def responses(input_text, **params):
    url = f"{ENDPOINT}/openai/v1/responses?api-version=preview"
    req = urllib.request.Request(
        url, data=json.dumps({"model": DEPLOYMENT, "input": input_text, **params}).encode(),
        headers={"Content-Type": "application/json", **HEADERS}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            return json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8','replace')[:400]}")


def rtext(resp):
    out = []
    for item in resp.get("output", []):
        for c in item.get("content", []):
            if c.get("type") in ("output_text", "text"):
                out.append(c.get("text", ""))
    return "".join(out).strip()


def measure(text):
    """이 텍스트가 실제로 몇 input 토큰인지 API 로 실측한다."""
    r = responses(text, max_output_tokens=16)
    return Usage.from_response(r, model=DEPLOYMENT).input_tokens


def ask(context, question):
    """컨텍스트로 질문하고 (답변, Usage) 를 돌려준다."""
    r = responses(f"{context}\n\n질문: {question}\n간결히 답하라.",
                  max_output_tokens=200, temperature=0)
    return rtext(r), Usage.from_response(r, model=DEPLOYMENT)


print("배포:", DEPLOYMENT, "· 준비 완료")

## 2. 실험 대상 — 실제로 컨텍스트를 잡아먹는 것

압축이라고 하면 흔히 긴 산문을 떠올리지만, 실무에서 컨텍스트를 잡아먹는 것은 대개
**기계가 만든 데이터**입니다.

- API 응답 JSON
- 로그·모니터링 출력
- 표를 텍스트로 옮긴 것

이런 데이터에는 **같은 말이 반복**됩니다. 그래서 무손실 압축의 효과가 큽니다.

In [ ]:
# 주문 내역 — 실제 API 응답처럼 pretty-print 된 JSON
ORDERS = [
    {"order_id": "A-1001", "status": "paid",      "amount": 32000, "currency": "KRW",
     "region": "KR", "channel": "app", "created_at": "2026-03-02", "vat_included": True},
    {"order_id": "A-1002", "status": "paid",      "amount": 15000, "currency": "KRW",
     "region": "KR", "channel": "app", "created_at": "2026-03-04", "vat_included": True},
    {"order_id": "A-1003", "status": "refunded",  "amount":  8000, "currency": "KRW",
     "region": "KR", "channel": "app", "created_at": "2026-03-05", "vat_included": True},
    {"order_id": "A-1004", "status": "shipping",  "amount": 47000, "currency": "KRW",
     "region": "KR", "channel": "app", "created_at": "2026-03-07", "vat_included": True},
    {"order_id": "A-1005", "status": "paid",      "amount": 21500, "currency": "KRW",
     "region": "KR", "channel": "app", "created_at": "2026-03-09", "vat_included": True},
    {"order_id": "A-1006", "status": "cancelled", "amount": 63000, "currency": "KRW",
     "region": "KR", "channel": "app", "created_at": "2026-03-11", "vat_included": True},
    {"order_id": "A-1007", "status": "paid",      "amount": 12800, "currency": "KRW",
     "region": "KR", "channel": "app", "created_at": "2026-03-12", "vat_included": True},
    {"order_id": "A-1008", "status": "shipping",  "amount": 39900, "currency": "KRW",
     "region": "KR", "channel": "app", "created_at": "2026-03-14", "vat_included": True},
]

V0 = json.dumps(ORDERS, ensure_ascii=False, indent=2)   # 원본: 보기 좋게 들여쓴 JSON

print(V0[:300], "…\n")
print(f"전체 {len(V0):,}자 · {len(ORDERS)}건")

## 3. 변환 1 — 정규화 (공백·구조 문자 제거)

JSON 을 사람이 보기 좋게 들여쓰면 **공백과 개행이 전부 토큰**이 됩니다.
모델은 들여쓰기가 없어도 JSON 을 읽습니다.

In [ ]:
V1 = json.dumps(ORDERS, ensure_ascii=False, separators=(",", ":"))   # minify

show_table(
    ["단계", "문자수", "샘플"],
    [["V0 pretty JSON", f"{len(V0):,}", V0[:46].replace(chr(10), "⏎") + "…"],
     ["V1 minified",    f"{len(V1):,}", V1[:46] + "…"]],
    align=["left", "right", "left"],
    note="들여쓰기·개행만 제거했다. 값은 하나도 안 건드렸으므로 완전한 무손실이다.",
)

## 4. 변환 2 — 구조 변환 (JSON → 표)

여기가 핵심입니다. JSON 배열은 **모든 행마다 키 이름을 반복**합니다.

```
{"order_id":"A-1001","status":"paid","amount":32000, ...}   ← 키가 8개
{"order_id":"A-1002","status":"paid","amount":15000, ...}   ← 또 8개
```

행이 8개면 키가 **64번** 나옵니다. 표로 바꾸면 키는 **헤더 1줄**이면 충분합니다.
행 수가 많을수록 이득이 커집니다.

In [ ]:
SEP = "|"

def to_table(rows, sep=SEP):
    """dict 배열 -> 구분자 표. 스키마가 동일할 때만 쓴다."""
    keys = list(rows[0].keys())
    lines = [sep.join(keys)]
    for r in rows:
        lines.append(sep.join(json.dumps(r[k], ensure_ascii=False)
                              if not isinstance(r[k], str) else r[k] for k in keys))
    return "\n".join(lines)


def from_table(text, sep=SEP):
    """표 -> dict 배열. 되돌릴 수 있어야 무손실이다."""
    lines = text.split("\n")
    keys = lines[0].split(sep)
    out = []
    for line in lines[1:]:
        vals = line.split(sep)
        row = {}
        for k, v in zip(keys, vals):
            try:
                row[k] = json.loads(v)      # 숫자·불리언 복원
            except Exception:
                row[k] = v
        out.append(row)
    return out


V2 = to_table(ORDERS)
print(V2[:220], "…\n")

# ★ 무손실 증명: 되돌려서 원본과 완전히 같은가
restored = from_table(V2)
print("왕복 복원 결과가 원본과 동일한가:", restored == ORDERS)

### 왜 이것이 "증명"일까요

`from_table(to_table(x)) == x` 가 참이면 **변환 과정에서 버려진 정보가 없다**는 뜻입니다.

- 압축률만 보고 "무손실인 것 같다"고 넘어가면 안 됩니다
- 되돌려서 대조하는 것이 유일하게 확실한 검증입니다
- 뒤에서 다룰 **손실** 압축은 이 검사에 실패합니다. 그것이 손실의 정의입니다

## 5. 변환 3 — 공통값 추출

표를 보면 `currency`, `region`, `channel`, `vat_included` 는 **모든 행이 같은 값**입니다.
8번 반복할 이유가 없으니 헤더로 한 번만 씁니다.

In [ ]:
def extract_common(rows, sep=SEP):
    """전 행이 동일한 필드를 헤더로 빼고, 나머지만 표로 만든다."""
    keys = list(rows[0].keys())
    common = {k: rows[0][k] for k in keys
              if len({json.dumps(r[k], ensure_ascii=False) for r in rows}) == 1}
    rest = [k for k in keys if k not in common]

    header = "공통: " + ", ".join(f"{k}={v}" for k, v in common.items())
    slim = [{k: r[k] for k in rest} for r in rows]
    return header + "\n" + to_table(slim, sep), common, rest


V3, COMMON, REST = extract_common(ORDERS)
print(V3[:260], "…\n")

# ★ 무손실 증명: 공통값을 되돌려 붙이면 원본과 같아야 합니다
restored3 = [{**COMMON, **r} for r in from_table(V3.split("\n", 1)[1])]
restored3 = [{k: r[k] for k in ORDERS[0]} for r in restored3]      # 키 순서 정렬
print("왕복 복원 결과가 원본과 동일한가:", restored3 == ORDERS)
print(f"\n공통으로 뺀 필드 {len(COMMON)}개: {list(COMMON)}")
print(f"행마다 남은 필드 {len(REST)}개: {REST}")

## 6. 여기까지의 실측

세 변환을 누적으로 적용하고 **실제 API 로 input 토큰을 측정**합니다. 추정하지 않습니다.

In [ ]:
# 각 단계가 무엇을 지웠는지 함께 적습니다. 숫자만 보면 무엇 때문에
# 줄었는지 알 수 없어서, 표에서 바로 읽히도록 설명을 붙입니다.
variants = [
    ("V0 원본 pretty JSON", V0, "—",
     f"들여쓰기 2칸으로 출력한 JSON. 주문 {len(ORDERS)}건이 기준선입니다"),
    ("V1 정규화 (minify)",  V1, "무손실",
     "들여쓰기·줄바꿈·구분자 뒤 공백을 제거했습니다"),
    ("V2 표 변환",          V2, "무손실",
     f"행마다 반복되던 키 이름 {len(ORDERS[0])}개를 헤더 한 줄로 올렸습니다"),
    ("V3 공통값 추출",      V3, "무손실",
     f"전 행이 같은 값인 {len(COMMON)}개 필드를 헤더에 한 번만 적었습니다"),
]

rows, base = [], None
for name, text, kind, what in variants:
    t = measure(text)
    base = base or t
    rows.append([name, what, kind, f"{len(text):,}", f"{t:,}",
                 f"{(1 - t/base):.0%}" if t != base else "—"])
    time.sleep(0.3)

show_table(
    ["단계", "무엇을 했나", "손실 여부", "문자수", "input 토큰", "절감률"],
    rows,
    align=["left", "left", "left", "right", "right", "right"],
    title="누적 적용 결과 (실측)",
    note="셋 다 앞 단계 위에 누적으로 적용했습니다. 정보를 하나도 안 버리고 "
         "얻은 절감이라, 되돌리면 원본이 그대로 나옵니다.",
)

print(f"공통으로 뺀 필드: {list(COMMON)}")
print(f"행마다 남은 필드: {REST}")

## 7. 행이 늘어나면 어떻게 될까요

표 변환의 이득은 **키 반복을 없애는 것**이므로 **행이 많을수록 유리**합니다.
반대로 행이 1~2개면 헤더 비용 때문에 **오히려 손해**일 수 있습니다.

이것은 압축 전반에 적용되는 원리입니다 — **"언제 쓰는가"가 "무엇을 쓰는가"보다 중요합니다.**

In [ ]:
def make_orders(n):
    return [{"order_id": f"A-{1000+i}", "status": ["paid", "shipping", "refunded"][i % 3],
             "amount": 10000 + i * 137, "currency": "KRW", "region": "KR",
             "channel": "app", "created_at": f"2026-03-{(i % 28) + 1:02d}",
             "vat_included": True}
            for i in range(n)]


rows = []
for n in (1, 2, 5, 20, 100):
    o = make_orders(n)
    j = json.dumps(o, ensure_ascii=False, separators=(",", ":"))
    t3, _, _ = extract_common(o)
    tj, tt = measure(j), measure(t3)
    rows.append([f"{n}건", f"{tj:,}", f"{tt:,}", f"{(1 - tt/tj):+.0%}"])
    time.sleep(0.3)

show_table(
    ["행 수", "minified JSON", "표+공통추출", "절감률"],
    rows,
    align=["right", "right", "right", "right"],
    title="행 수에 따른 표 변환 효과",
    note="행이 적으면 이득이 거의 없거나 손해다. 규칙: 반복이 있어야 압축이 산다.",
)

## 8. 변환 4 — 화이트리스트 컷 (여기부터 손실)

지금까지는 되돌릴 수 있었습니다. 이제 **정말로 버립니다.**

질문에 필요 없는 필드를 제거하는 것인데, 이 순간 **무손실이 아니게 됩니다.**
`from_table` 로 되돌려도 버린 필드는 돌아오지 않습니다.

In [ ]:
QUESTION = "환불(refunded) 상태인 주문의 주문번호와 금액은?"
KEEP = ["order_id", "status", "amount"]      # 질문에 필요한 것만

cut = [{k: r[k] for k in KEEP} for r in ORDERS]
V4 = to_table(cut)

# ★ 손실 확인: 되돌려도 원본과 같지 않습니다
restored4 = from_table(V4)
print("왕복 복원 결과가 원본과 동일한가:", restored4 == ORDERS)
print("→ False 가 정상입니다. 이것이 '손실'의 정의입니다.\n")

lost = set(ORDERS[0]) - set(KEEP)
print(f"버린 필드: {sorted(lost)}")
print(f"남긴 필드: {KEEP}")
print(f"\n{V4[:150]}…")

### 손실 압축의 진짜 위험

버린 필드가 **이번 질문**에 필요 없다는 것은 확인했습니다. 문제는 **다음 질문**입니다.

| 질문 | V3(무손실) | V4(필드 컷) |
|---|---|---|
| "환불 건의 금액은?" | 답합니다 | 답합니다 |
| "부가세 포함 금액인가?" | 답합니다 | **답하지 못합니다** — `vat_included` 를 버렸습니다 |
| "3월 10일 이후 주문은?" | 답합니다 | **답하지 못합니다** — `created_at` 을 버렸습니다 |

즉 **화이트리스트 컷은 질문을 미리 안다고 가정합니다.**
질문이 고정된 파이프라인에서는 안전하지만, 자유 질의에서는 위험합니다.

아래에서 실제로 확인합니다.

In [ ]:
probe_questions = [
    ("이번 질문",   QUESTION),
    ("다른 질문 1", "이 주문들의 금액은 부가세가 포함된 값인가?"),
    ("다른 질문 2", "2026년 3월 10일 이후에 생성된 주문번호를 모두 알려줘."),
]

rows = []
for label, q in probe_questions:
    a3, _ = ask(V3, q)      # 무손실
    a4, _ = ask(V4, q)      # 필드 컷
    rows.append([label, q[:26] + "…", a3[:34] + "…", a4[:34] + "…"])
    time.sleep(0.3)

show_table(
    ["구분", "질문", "V3 무손실 답변", "V4 필드컷 답변"],
    rows,
    align=["left", "left", "left", "left"],
    note="V4 는 이번 질문엔 잘 답하지만, 버린 필드를 묻는 질문에는 답하지 못한다.",
)

## 9. 정답이 바뀌지 않았는지 확인

무손실 압축이라면 **모델의 답이 달라질 이유가 없습니다.**
V0 ~ V3 에 같은 질문을 던져 답이 일치하는지 봅니다.

이것이 무손실 압축의 마지막 관문입니다. 이론상 무손실이어도
**모델이 표 형식을 JSON 만큼 잘 읽는다는 보장은 없기 때문**입니다.

In [ ]:
EXPECT = ["A-1003", "8000"]      # 반드시 나와야 하는 문자열

rows = []
for name, text in [("V0 원본 JSON", V0), ("V1 minified", V1),
                   ("V2 표", V2), ("V3 표+공통추출", V3), ("V4 필드컷", V4)]:
    ans, u = ask(text, QUESTION)
    hit = [e for e in EXPECT if e.replace(",", "") in ans.replace(",", "")]
    rows.append([name, f"{u.input_tokens:,}", f"{len(hit)}/{len(EXPECT)}",
                 "OK" if len(hit) == len(EXPECT) else "실패", ans[:40] + "…"])
    time.sleep(0.3)

show_table(
    ["형식", "input", "정답요소", "판정", "답변"],
    rows,
    align=["left", "right", "right", "left", "left"],
    title=f"같은 질문: {QUESTION}",
    note="형식을 바꿔도 답이 유지되어야 무손실 압축이 '실제로' 성립한다.",
)

## 10. 캐시와 부딪히는 지점

01번 8절에서 확인했습니다 — **접두부가 바뀌면 캐시가 전량 미스**가 납니다.

압축은 텍스트를 바꾸는 일이므로 **압축 대상을 어디에 두느냐**가 중요합니다.

| 배치 | 결과 |
|---|---|
| 고정 시스템 프롬프트를 압축 | 접두부가 바뀝니다 → **캐시 파괴** |
| 뒤쪽 가변 데이터만 압축 | 접두부는 그대로입니다 → **캐시 유지 + 토큰 절감** |

즉 무손실 압축이라도 **아무 데나 적용하면 손해**입니다.
아래에서 두 배치를 실측으로 비교합니다.

In [ ]:
import uuid

RUN_ID = uuid.uuid4().hex[:8]        # 실행마다 찬 캐시에서 시작 (01번 8절과 동일한 이유)

# 1,024토큰을 넘기는 고정 시스템 지침
POLICY = (f"[run={RUN_ID}] 당신은 주문 관리 상담원입니다. 아래 규칙을 따르십시오.\n"
          + "".join(f"규칙{i}. 답변은 근거 데이터에 있는 값만 사용하며, "
                    f"추측하지 않고, 단위와 통화를 그대로 표기한다.\n"
                    for i in range(1, 61)))


def trial(label, build, turns=3):
    rows, hits = [], []
    for t in range(1, turns + 1):
        r = responses(build(), max_output_tokens=32)
        u = Usage.from_response(r, model=DEPLOYMENT)
        hits.append(u.cache_hit_rate)
        rows.append([f"{t}회차", f"{u.input_tokens:,}", f"{u.cached_tokens:,}",
                     f"{u.billed_input:,}", f"{u.cache_hit_rate:.1%}"])
        time.sleep(1.5)
    show_table(["회차", "입력", "캐시적중", "과금입력", "적중률"], rows,
               align=["left", "right", "right", "right", "right"], title=label)
    return rows[-1]


# A) 고정부는 그대로 두고 뒤쪽 데이터만 압축  -> 캐시 유지
a = trial("A) 고정 정책(앞) + 압축된 데이터(뒤)",
          lambda: f"{POLICY}\n\n[주문]\n{V3}\n\n질문: {QUESTION}")

# B) 정책까지 매번 다르게 만든 경우          -> 캐시 파괴
b = trial("B) 앞쪽이 매번 바뀌는 경우",
          lambda: f"[ts={time.time_ns()}]\n{POLICY}\n\n[주문]\n{V3}\n\n질문: {QUESTION}")

show_table(
    ["배치", "마지막 회차 과금입력", "적중률"],
    [["A) 고정부 보존", a[3], a[4]], ["B) 접두부 변경", b[3], b[4]]],
    align=["left", "right", "right"],
    title="결론",
    note="같은 데이터를 같은 만큼 압축해도, 배치에 따라 과금 입력이 크게 갈립니다.",
)

## 11. 언제 이득이고 언제 아닐까요

### 이득이 큰 경우

- **반복이 많을 때** — 같은 키와 값이 여러 번 나오는 경우 (JSON 배열, 로그, 표)
- **행이 많을 때** — 표 변환은 행 수에 비례해 이득이 커집니다
- **기계가 만든 데이터일 때** — 사람이 쓴 산문은 이미 중복이 적습니다

### 이득이 없거나 손해인 경우

- **행이 1~2개일 때** — 헤더 비용이 더 큽니다
- **이미 짧을 때** — 1,024토큰 미만이면 캐시도 안 걸리고 절감액도 미미합니다
- **스키마가 제각각일 때** — 표로 만들 수 없습니다
- **캐시가 잘 맞던 접두부일 때** — 건드리면 손해입니다 (10절)

### 실행 순서

| 순서 | 할 일 | 되돌릴 수 있나 |
|---|---|---|
| 1 | 정규화 (공백·주석 제거) | 예 |
| 2 | 구조 변환 (JSON→표) | 예 |
| 3 | 공통값 추출 | 예 |
| 4 | **여기서 목표를 달성했는지 확인** | — |
| 5 | 화이트리스트 컷 | **아니오** |
| 6 | 손실 압축 (요약·프루닝) | **아니오** |

> **4번이 핵심입니다.** 1~3 만으로 예산 안에 들어오면 5~6 은 하지 않습니다.
> 위험을 감수할 이유가 없습니다.

## 12. 정리

| 배운 것 | 근거 |
|---|---|
| 무손실만으로도 상당히 줄어듭니다 | 6절 실측 |
| 무손실은 **되돌려서 증명**해야 합니다 | 4·5절 왕복 검증 |
| 표 변환의 이득은 **행 수에 비례**합니다 | 7절 |
| 화이트리스트 컷은 **질문을 안다고 가정**합니다 | 8절 — 다른 질문에는 답하지 못합니다 |
| 형식이 바뀌어도 **답은 같아야** 합니다 | 9절 |
| 무손실이어도 **배치를 잘못하면 손해**입니다 | 10절 캐시 |

### 이 노트북이 남긴 숙제

9절에서 답을 문자열 포함 여부로 확인했습니다. 케이스가 5개라 눈으로 봤지만,
**압축 파라미터를 수십 개 조합으로 돌리면 눈으로 볼 수 없습니다.**

그래서 다음이 필요합니다.

- **골든셋** — "무엇이 깨지면 실패인가"를 미리 정의한 케이스 모음
- **`survival` 지표** — 반드시 살아야 할 문자열이 압축 후에도 있는지 자동으로 검사

여기서 만든 `to_table` 과 `extract_common` 은 나중에 `labs/03-lossless-structure/` 로 옮길 예정입니다.